In [139]:
import pandas as p
import pickle
import pickletools

%pylab inline

Populating the interactive namespace from numpy and matplotlib


In [35]:
df = p.read_pickle('/home/matej/tcm16.pkl')

In [42]:
strainmap = {}
for allele, blid, orf, name, aliases in Strain.objects.select_related('gene').values_list(
    'allele', 'boonelab_id', 'gene__orf', 'gene__name', 'gene__aliases'):
    label = (allele or name or orf).lower()
    strainmap[orf.lower()] = label
    strainmap[blid.lower()] = label
    if allele:
        strainmap[allele.lower()] = label
    if name:
        strainmap[name.lower()] = label
    for a in aliases:
        if a not in strainmap:
            strainmap[a.lower()] = label

In [60]:
sheets = p.read_excel('/home/matej/Documents/SGS1_GI_PI_scores_6col.xlsx')

In [61]:
sheets.loc[:, 'Label'] = sheets.Label.apply(lambda x: strainmap.get(x.lower(), x))
sheets.index = sheets.Label

In [66]:
sheets = sheets.loc[~sheets.duplicated(subset='Label')]

In [69]:
sheets = sheets.reindex(index=df.columns)

In [81]:
corrs = {}
for input_ds in sheets.columns[1:2]:
    corrs[input_ds] = df.corrwith(sheets[input_ds], axis=1)

In [87]:
x = corrs['PI_Min']

In [147]:
p.DataFrame(corrs)

,PI_Min
6231,0.031620
6232,0.041250
2679,0.009177
4166,0.020359
4217,0.012814
4380,0.035593
2663,0.033185
92,0.008592
1503,0.003676
3741,-0.011269


In [100]:
mak3 = StrainData.objects.filter(strain__gene__name='MAK3', dataset__name='Living')[0]

In [103]:
dataset = Dataset.objects.get(name='Living')

In [105]:
arrays = [a.label() for a  in dataset.arrays.select_related('gene')]

In [110]:
p.DataFrame(mak3.scores, index=arrays).to_csv('/home/matej/mak3.csv')

In [111]:
def strain_map():
    strainmap = {}
    for allele, blid, orf, name, aliases in Strain.objects.select_related('gene').values_list(
            'allele', 'boonelab_id', 'gene__orf', 'gene__name', 'gene__aliases'):
        label = (allele or name or orf).lower()
        strainmap[orf.lower()] = label
        strainmap[blid.lower()] = label
        if allele:
            strainmap[allele.lower()] = label
        if name:
            strainmap[name.lower()] = label
        for a in aliases:
            if a not in strainmap:
                strainmap[a.lower()] = label
    return strainmap

In [112]:
%%timeit
strain_map()

81.1 ms ± 2.49 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [113]:
sm = strain_map()

In [118]:
pickle.dump(sm, open('/home/matej/sm.pkl', 'wb'))

In [120]:
%%timeit
pickle.load(open('/home/matej/sm.pkl', 'rb'))

7.41 ms ± 110 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [127]:
def strain_map2():
    strainmap = {}
    for allele, blid, orf, name, aliases in Strain.objects.select_related('gene').values_list(
            'allele', 'boonelab_id', 'gene__orf', 'gene__name', 'gene__aliases'):
        label = (allele or name or orf).lower()
        
        strainmap[blid.lower()] = label
        
        if orf.lower() != label:
            strainmap[orf.lower()] = label
        
        if allele and allele.lower() != label:
            strainmap[allele.lower()] = label
        
        if name and name.lower() != label:
            strainmap[name.lower()] = label
        
        for a in aliases:
            if a not in strainmap and len(a) < 16:
                strainmap[a.lower()] = label
    return strainmap

In [128]:
%%timeit
strain_map2()

79.1 ms ± 1.38 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [129]:
sm2 = strain_map2()

In [134]:
pickle.dump(sm2, open('/home/matej/sm2v1.pkl', 'wb'), protocol=1)
pickle.dump(sm2, open('/home/matej/sm2v2.pkl', 'wb'), protocol=2)
pickle.dump(sm2, open('/home/matej/sm2v3.pkl', 'wb'), protocol=3)
pickle.dump(sm2, open('/home/matej/sm2v4.pkl', 'wb'), protocol=4)

In [135]:
%%timeit
pickle.load(open('/home/matej/sm2v1.pkl', 'rb'))

5.18 ms ± 137 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [136]:
%%timeit
pickle.load(open('/home/matej/sm2v2.pkl', 'rb'))

5.34 ms ± 462 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [137]:
%%timeit
pickle.load(open('/home/matej/sm2v3.pkl', 'rb'))

4.91 ms ± 92.7 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [138]:
%%timeit
pickle.load(open('/home/matej/sm2v4.pkl', 'rb'))

4.4 ms ± 113 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [132]:
pickle.load(open('/home/matej/sm2.pkl', 'rb'))

{'tsq508': 'tfc3-g349e',
 'yal001c': 'tfc3-g349e',
 'tfc3': 'tfc3-g349e',
 'tau 138': 'tfc3-g349e',
 'tsv115': 'tfc3-g349e',
 'fun24': 'tfc3-g349e',
 'sn273': 'vps8',
 'yal002w': 'vps8',
 'vpl8': 'vps8',
 'vpt8': 'vps8',
 'fun15': 'vps8',
 'sn2532': 'fun14',
 'yal008w': 'fun14',
 'mcp3': 'fun14',
 'sn1073': 'spo7',
 'yal009w': 'spo7',
 'sn2699': 'mdm10',
 'yal010c': 'mdm10',
 'fun37': 'mdm10',
 'sn1778': 'swc3',
 'yal011w': 'swc3',
 'swc1': 'swc3',
 'sn4470': 'cys3',
 'yal012w': 'cys3',
 'str1': 'cys3',
 'fun35': 'cys3',
 'cyi1': 'cys3',
 'sn2007': 'dep1',
 'yal013w': 'dep1',
 'fun54': 'dep1',
 'sn4677': 'syn8',
 'yal014c': 'syn8',
 'syntaxin': 'syn8',
 'slt2': 'syn8',
 'uip2': 'syn8',
 'sn1277': 'ntg1',
 'yal015c': 'ntg1',
 'ogg2': 'ntg1',
 'scr1': 'ntg1',
 'fun33': 'ntg1',
 'sn4628': 'yal016c-b',
 'sn2937': 'psk1',
 'yal017w': 'psk1',
 'fun31': 'psk1',
 'sn3084': 'lds1',
 'yal018c': 'lds1',
 'sn1217': 'fun30',
 'yal019w': 'fun30',
 'sn1168': 'ats1',
 'yal020c': 'ats1',
 'kti13': 'ats

In [142]:
optp = pickletools.optimize(pickle.dumps(sm2, protocol=4))

In [144]:
with open('/home/matej/sm2opt.pkl', 'wb') as fout:
    fout.write(optp)

In [146]:
%%timeit
pickle.load(open('/home/matej/sm2opt.pkl', 'rb'))

4.01 ms ± 63.8 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [148]:
x = p.DataFrame(corrs)

In [155]:
x

,PI_Min
6231,0.031620
6232,0.041250
2679,0.009177
4166,0.020359
4217,0.012814
4380,0.035593
2663,0.033185
92,0.008592
1503,0.003676
3741,-0.011269


In [156]:
sheets.PI_Min.corr(sheets.PI_Log, )

0.92191037370945295